In [ ]:
import sys
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
MODEL = os.getenv("MODEL")
BASE_URL = os.getenv("OLLAMA_BASE_URL")

REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT / "src"))

from wraith.parsers.cowrie_parser import resolve_log_files, parse_sessions
from wraith.llm.session_assessment import analyze_session
from wraith.enrichment.triage import triage_batch
from wraith.audit.audit_log import log_batch

In [ ]:
RAW_LOG_DIR = REPO_ROOT / "data" / "raw"

# Scope this run to just the last day or two while iterating - set to
# None to process everything currently on disk.
SINCE_DATE = "2026-08-22"

log_files = [
    path for path in resolve_log_files(RAW_LOG_DIR)
    if path.name == "cowrie.json" or path.name.removeprefix("cowrie.json.") >= SINCE_DATE
]
print(f"{len(log_files)} file(s) selected: {[f.name for f in log_files]}")

summaries = parse_sessions(log_files)
print(f"{len(summaries)} sessions parsed")

In [ ]:
for s in summaries:
    print(
        f"{s.session_id}  src_ip={s.src_ip}  "
        f"attempts={s.login_attempts}  success={s.login_success}  "
        f"commands={len(s.commands)}  downloads={len(s.files_downloaded)}"
    )

In [ ]:
sessions_with_commands = [s for s in summaries if s.commands]
print(f"{len(sessions_with_commands)} session(s) with commands entered")

if sessions_with_commands:
    example = sessions_with_commands[0]
    print(example.model_dump_json(indent=2))

In [ ]:
raw_line_count = 0
for f in log_files:
    with open(f) as fh:
        raw_line_count += sum(1 for _ in fh)

print(f"raw event lines across all matched files: {raw_line_count}")
print(f"sessions grouped: {len(summaries)}")

## LLM triage stage

Everything above this point is deterministic parsing only - no model
involved. From here: candidate ATT&CK enrichment, trivial-session
filtering, LLM assessment for anything non-trivial, IoC correlation on
flagged sessions, and an audit record for every decision (LLM or
rule-based skip alike).

In [ ]:
# Runs the full pipeline per session:
#   trivial sessions -> auto-benign, no LLM call
#   everything else  -> enrich_session() candidates -> analyze_session() via Ollama
#   suspicious/malicious results -> correlate_ioc() lookup
#
# This will make one HTTP call per non-trivial session to BASE_URL - expect
# it to take a while across two days of real traffic. No progress output
# from triage_batch() itself, so the cell will look idle until it's done.

results = triage_batch(summaries, analyze_session, BASE_URL, MODEL)

print(f"{len(results)} session(s) triaged")

method_counts = {}
for r in results:
    method_counts[r["method"]] = method_counts.get(r["method"], 0) + 1

for method, count in method_counts.items():
    print(f"  {count:>4}  {method}")

In [ ]:
classification_counts = Counter(r["classification"] for r in results)

for classification, count in classification_counts.items():
    print(f"{classification:>12}: {count}")

In [ ]:
flagged_results = [r for r in results if r["classification"] in ("suspicious", "malicious")]
print(f"{len(flagged_results)} session(s) classified suspicious/malicious")

for r in flagged_results:
    print(f"\n--- {r['session_id']}  ({r['src_ip']}) ---")
    print(f"classification={r['classification']}  confidence={r['confidence']}")
    if r["assessment"]:
        print(r["assessment"].model_dump_json(indent=2))
    if r["ioc_history"]:
        ioc = r["ioc_history"]
        print(f"IoC correlation: {len(ioc.related_session_ids)} related session(s) "
              f"({ioc.related_session_ids}), any_login_success={ioc.any_login_success}, "
              f"total_commands={ioc.total_commands}, total_downloads={ioc.total_downloads}")

## Audit trail

Writes one record per session to `logs/audit_trail.jsonl` - rule-based
skips and LLM assessments alike. Records flagged by the contextually-blind
heuristic (excluded technique justified via isolation-language on a
session with other included techniques) print immediately below, worth
reviewing by hand rather than trusting the flag alone.

In [ ]:
records = log_batch(results, model=MODEL)

print(f"\n{len(records)} record(s) written to audit trail")